# 03 — CNN en transfert learning (Phase 4)

ResNet18 pré-entraîné ImageNet, en deux temps : **phase A** (backbone gelé, seule la tête apprend) puis **phase B** (fine-tuning de `layer3`/`layer4` + tête, learning rate plus faible). Choix du seuil de décision sur `val_ood` pour viser un rappel élevé (un faux négatif = une métastase manquée, voir `outputs/rapport/01_contexte.md`).

**Sur Colab** : *Exécution → Modifier le type d'exécution → GPU (T4)*, avant de lancer ce notebook. `QUICK = True` ci-dessous permet de vérifier rapidement que tout fonctionne sur un petit sous-ensemble avant de lancer le run complet.

Voir `PLAN.md` (Phase 4) et `outputs/rapport/04_cnn.md`.

In [ ]:
# Colab/Kaggle : clone le depot si besoin et place le repertoire de travail dans notebooks/
# (sans effet en local : le depot est deja la, donc rien n'est clone ni deplace)
import os, subprocess, sys

REPO_URL = "https://github.com/YoucefLgr/Breast_Cancer.git"
EN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if EN_COLAB or EN_KAGGLE:
    repo_dir = "/content/Breast_Cancer" if EN_COLAB else "/kaggle/working/Breast_Cancer"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    os.chdir(os.path.join(repo_dir, "notebooks"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

print("Repertoire de travail :", os.getcwd())

In [ ]:
# Paramètres
QUICK = False   # True = sous-ensemble réduit, pour vérifier que tout tourne avant le run complet

import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # évite un plantage connu (OpenMP) sur certains environnements CPU

import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils import load_config, set_seed, get_device, detect_environment, evaluer_par_hopital, save_metrics, make_generator
from src.data import charger_sous_echantillon, PatchDataset, transform_entrainement, transform_evaluation, HOPITAL_LABEL
from src.model import construire_modele, geler_backbone, degeler_layers_superieures, compter_parametres
from src.train import entrainer, predire_probas, choisir_seuil

CFG = load_config()
SEED = CFG["seed"]
set_seed(SEED)
DEVICE = get_device()
print("Environnement :", detect_environment(), "| device :", DEVICE)

## 1. Données

Mêmes trois splits que la Phase 3 : entraînement sur `train` (H0/3/4), tous les choix (seuil, épaisseur du fine-tuning) sur `val_ood` (H1). `test` (H2) n'est pas chargé ici.

In [ ]:
df = charger_sous_echantillon(CFG)
if QUICK:
    morceaux = [g.sample(min(len(g), 60), random_state=SEED)
                for _, g in df.groupby(["split", "center", "label"])]
    df = pd.concat(morceaux, ignore_index=True)
    print("Mode QUICK : sous-ensemble réduit à", len(df), "patchs")

df_train = df[df.split == "train"].reset_index(drop=True)
df_val = df[df.split == "val_ood"].reset_index(drop=True)
df_idval = df[df.split == "id_val"].reset_index(drop=True)
print(f"train={len(df_train):,}  val_ood={len(df_val):,}  id_val={len(df_idval):,}")

gen = make_generator(SEED)
BS = 32 if QUICK else CFG["modele"]["batch_size"]
dl_train = DataLoader(PatchDataset(df_train, transform_entrainement(CFG)), batch_size=BS,
                      shuffle=True, generator=gen, num_workers=0)
dl_val = DataLoader(PatchDataset(df_val, transform_evaluation(CFG)), batch_size=BS, shuffle=False)
dl_idval = DataLoader(PatchDataset(df_idval, transform_evaluation(CFG)), batch_size=BS, shuffle=False)

## 2. Modèle, phase A (tête seule)

In [ ]:
modele = construire_modele(CFG["modele"]["architecture"]).to(DEVICE)
geler_backbone(modele)
total, entrainables = compter_parametres(modele)
print(f"Phase A : {entrainables:,} / {total:,} paramètres entraînables ({entrainables / total:.1%})")

epochs_tete = 1 if QUICK else CFG["modele"]["epochs_tete"]
res_a = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE,
                  epochs=epochs_tete, lr=CFG["modele"]["lr_tete"],
                  weight_decay=CFG["modele"]["weight_decay"], patience=2, verbeux=True)
print("Meilleur AUC val_ood (phase A) :", round(res_a["meilleur_auc_val_ood"], 4))

## 3. Phase B (fine-tuning layer3/layer4 + tête)

Learning rate plus faible pour ne pas détruire les poids ImageNet déjà utiles — on ajuste, on ne réapprend pas depuis zéro.

In [ ]:
degeler_layers_superieures(modele)
total, entrainables = compter_parametres(modele)
print(f"Phase B : {entrainables:,} / {total:,} paramètres entraînables ({entrainables / total:.1%})")

epochs_ft = 1 if QUICK else CFG["modele"]["epochs_finetune"]
res_b = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE,
                  epochs=epochs_ft, lr=CFG["modele"]["lr_finetune"],
                  weight_decay=CFG["modele"]["weight_decay"], patience=2,
                  checkpoint_path="../outputs/models/resnet18_phaseB.pt", verbeux=True)
print("Meilleur AUC val_ood (phase B) :", round(res_b["meilleur_auc_val_ood"], 4))

In [ ]:
hist = pd.DataFrame(res_a["historique"]).assign(phase="A")._append(
    pd.DataFrame(res_b["historique"]).assign(phase="B"), ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for phase, c in [("A", "tab:blue"), ("B", "tab:orange")]:
    h = hist[hist.phase == phase]
    axes[0].plot(range(1, len(h) + 1), h.perte_train, "o-", color=c, label=f"phase {phase}")
    axes[1].plot(range(1, len(h) + 1), h.auc_val_ood, "o-", color=c, label=f"phase {phase}")
axes[0].set_title("Perte (train)"); axes[0].set_xlabel("époque"); axes[0].legend()
axes[1].set_title("ROC-AUC (val_ood, H1)"); axes[1].set_xlabel("époque"); axes[1].legend()
plt.tight_layout(); plt.savefig("../outputs/figures/05_courbes_cnn.png", dpi=130); plt.show()

## 4. Comparaison avec la référence simple et avec le seuil par défaut

In [ ]:
proba_val = predire_probas(modele, dl_val, DEVICE)
proba_idval = predire_probas(modele, dl_idval, DEVICE)

print("-- Phase B, val_ood (H1), seuil 0,5 --")
t_val_05 = evaluer_par_hopital(df_val.label.values, proba_val, df_val.center.values, 0.5, HOPITAL_LABEL)
display(t_val_05)
print("-- Phase B, id_val (H0/3/4, repère biaisé), seuil 0,5 --")
t_idval_05 = evaluer_par_hopital(df_idval.label.values, proba_idval, df_idval.center.values, 0.5, HOPITAL_LABEL)
display(t_idval_05)

## 5. Choix du seuil orienté rappel (sur `val_ood` uniquement)

In [ ]:
rappel_cible = CFG["evaluation"]["rappel_cible"]
choix_seuil = choisir_seuil(df_val.label.values, proba_val, rappel_cible)
print(choix_seuil)

print(f"-- Phase B, val_ood (H1), seuil ajusté {choix_seuil['seuil']} (cible rappel >= {rappel_cible}) --")
t_val_ajuste = evaluer_par_hopital(df_val.label.values, proba_val, df_val.center.values,
                                   choix_seuil["seuil"], HOPITAL_LABEL)
display(t_val_ajuste)

## 6. Sauvegarde des métriques

In [ ]:
metrics = {
    "quick_mode": QUICK,
    "architecture": CFG["modele"]["architecture"],
    "protocole": {"entrainement": "train (H0/3/4)", "selection": "val_ood (H1)",
                  "hopital_2": "non utilise (reserve Phase 6)"},
    "historique_phase_A": res_a["historique"],
    "historique_phase_B": res_b["historique"],
    "meilleur_auc_val_ood": res_b["meilleur_auc_val_ood"],
    "metriques_seuil_0.5": {"val_ood": t_val_05.to_dict(orient="records"),
                            "id_val": t_idval_05.to_dict(orient="records")},
    "choix_seuil": choix_seuil,
    "metriques_seuil_ajuste": {"val_ood": t_val_ajuste.to_dict(orient="records")},
    "reference_phase3_auc_val_ood": 0.898,
}
print(save_metrics(metrics, "outputs/metrics/04_cnn.json"))

## 7. Lecture des résultats

Voir `outputs/rapport/04_cnn.md` : comparaison au repère de la Phase 3 (AUC 0,898 sur `val_ood`), effet du fine-tuning (phase A vs B), effet du seuil ajusté sur le rappel et la précision. Le modèle choisi ici (checkpoint `outputs/models/resnet18_phaseB.pt`) sera celui comparé aux variantes anti-shift de la Phase 5, puis évalué une seule fois sur l'hôpital 2 en Phase 6.